# nf-core/scrnaseq Sandwich Wrapper

**Genesis Workbench** — Zero-Fork Nextflow Orchestration

This notebook implements a **sandwich wrapper pattern** that keeps `nf-core/scrnaseq` as a completely untouched black box. All Eisai-specific logic lives in the "bread" — Databricks-native pre-flight and post-flight layers — so that upgrading nf-core requires only bumping the version pin.

```
┌─────────────────────────────────────────────────────┐
│             PRE-FLIGHT (Databricks-native)           │
│  ┌───────────┐ ┌──────────┐ ┌────────────────────┐  │
│  │ Validate  │ │ Build    │ │ Generate           │  │
│  │ inputs &  │→│ sample   │→│ eisai.config       │  │
│  │ stage     │ │ sheet    │ │ (institutional     │  │
│  │ FASTQs    │ │          │ │  config overlay)   │  │
│  └───────────┘ └──────────┘ └────────────────────┘  │
│  Register run in Delta audit table                   │
├─────────────────────────────────────────────────────┤
│           nf-core/scrnaseq (BLACK BOX)              │
│                                                      │
│  nextflow run nf-core/scrnaseq                       │
│    -r <pinned_version>                               │
│    -c eisai.config        ← institutional overlay    │
│    -profile conda                                    │
│    --input samplesheet.csv                           │
│    --outdir /Volumes/.../output                      │
│                                                      │
│  ⚠️  ZERO modifications to the pipeline itself       │
├─────────────────────────────────────────────────────┤
│            POST-FLIGHT (Databricks-native)           │
│  ┌───────────┐ ┌──────────┐ ┌────────────────────┐  │
│  │ Parse NF  │ │ Run QC   │ │ Ingest outputs     │  │
│  │ execution │→│ gates on │→│ to UC Volumes &    │  │
│  │ trace →   │ │ alignment│ │ Delta tables       │  │
│  │ Delta     │ │ quality  │ │                    │  │
│  └───────────┘ └──────────┘ └────────────────────┘  │
│  Archive MultiQC │ Trigger scanpy │ MLflow logging   │
└─────────────────────────────────────────────────────┘
```

### Why This Pattern?

| Concern | Where It Lives | nf-core Touch? |
|---------|---------------|----------------|
| Input validation | Pre-flight | None |
| Resource tuning | `eisai.config` process selectors | None |
| Custom containers | `eisai.config` `withName:` overrides | None |
| publishDir routing to Volumes | `eisai.config` | None |
| Alignment QC gates | Post-flight | None |
| Execution metrics → Delta | Post-flight (trace CSV parse) | None |
| Downstream scanpy trigger | Post-flight (Jobs API) | None |

**Version bump workflow**: Change `pipeline_version` widget → review nf-core changelog for renamed params → adjust `eisai.config` if any process names changed → done (~30 min).

In [0]:
# Widget parameters — set by Databricks Jobs API from Streamlit form
# Existing params (compatible with run_nextflow_scrnaseq)
dbutils.widgets.text("fastq_dir", "", "FASTQ Directory")
dbutils.widgets.text("output_dir", "", "Output Directory")
dbutils.widgets.dropdown("aligner", "starsolo", ["starsolo", "salmon", "kallisto", "cellranger"], "Aligner")
dbutils.widgets.dropdown("genome", "GRCh38", ["GRCh38", "GRCm39", "GRCm38"], "Reference Genome")
dbutils.widgets.dropdown("chemistry", "auto", ["auto", "10XV2", "10XV3", "10XV3HT", "10XV5"], "10x Chemistry")
dbutils.widgets.text("sample_name", "", "Sample Name (optional)")
dbutils.widgets.text("pipeline_version", "2.7.1", "nf-core/scrnaseq Version")
dbutils.widgets.text("extra_args", "", "Extra Nextflow Args")

# Sandwich-specific params
dbutils.widgets.dropdown("qc_gate_enabled", "true", ["true", "false"], "Enable Alignment QC Gates")
dbutils.widgets.text("min_cells", "200", "Min Cells (QC gate)")
dbutils.widgets.text("min_median_genes", "500", "Min Median Genes/Cell (QC gate)")
dbutils.widgets.dropdown("trigger_scanpy", "false", ["true", "false"], "Auto-trigger Scanpy on QC Pass")
dbutils.widgets.text("project_name", "", "Project Name (for audit trail)")

# Resolve all widgets
fastq_dir = dbutils.widgets.get("fastq_dir")
output_dir = dbutils.widgets.get("output_dir")
aligner = dbutils.widgets.get("aligner")
genome = dbutils.widgets.get("genome")
chemistry = dbutils.widgets.get("chemistry")
sample_name = dbutils.widgets.get("sample_name") or None
pipeline_version = dbutils.widgets.get("pipeline_version")
extra_args = dbutils.widgets.get("extra_args") or None
qc_gate_enabled = dbutils.widgets.get("qc_gate_enabled") == "true"
min_cells = int(dbutils.widgets.get("min_cells"))
min_median_genes = int(dbutils.widgets.get("min_median_genes"))
trigger_scanpy = dbutils.widgets.get("trigger_scanpy") == "true"
project_name = dbutils.widgets.get("project_name") or "unnamed"

import time
run_id = f"{project_name}_{int(time.time())}"

print(f"\n{'='*60}")
print(f"  Sandwich Wrapper Run: {run_id}")
print(f"{'='*60}")
print(f"  FASTQ dir:      {fastq_dir}")
print(f"  Output dir:     {output_dir}")
print(f"  Aligner:        {aligner}")
print(f"  Genome:         {genome}")
print(f"  Pipeline:       nf-core/scrnaseq v{pipeline_version}")
print(f"  QC gates:       {'ON' if qc_gate_enabled else 'OFF'}")
print(f"  Auto scanpy:    {'ON' if trigger_scanpy else 'OFF'}")
print(f"{'='*60}")

---
## 🔵 Layer 1 — Pre-Flight (Databricks-Native)

Everything before `nextflow run`. Validates inputs, builds the samplesheet, generates the institutional config overlay, and registers the run in a Delta audit table. **Zero nf-core modifications.**

In [0]:
import os, sys, glob, subprocess, json
from datetime import datetime, timezone

# ── Validate Nextflow is installed ──
try:
    nxf_out = subprocess.check_output(["nextflow", "-version"], stderr=subprocess.STDOUT, text=True)
    nxf_version = [l for l in nxf_out.strip().split("\n") if "version" in l.lower()]
    print(f"\u2705 Nextflow: {nxf_version[0].strip() if nxf_version else nxf_out.strip().split(chr(10))[0]}")
except FileNotFoundError:
    raise RuntimeError(
        "\u274c Nextflow not found. Cluster needs init script: "
        "/Volumes/dhbl_discovery_us_dev/genesis_schema/scanpy_reference/init_scripts/install_nextflow.sh"
    )

# ── Validate FASTQ directory ──
assert fastq_dir and os.path.isdir(fastq_dir), f"\u274c FASTQ directory not found: {fastq_dir}"
assert output_dir, "\u274c Output directory must be specified"
os.makedirs(output_dir, exist_ok=True)

fastq_files = []
for pattern in ["**/*.fastq.gz", "**/*.fastq", "**/*.fq.gz", "**/*.fq"]:
    fastq_files.extend(glob.glob(os.path.join(fastq_dir, pattern), recursive=True))

assert fastq_files, f"\u274c No FASTQ files found in {fastq_dir}"

# Quick read-pair sanity check
r1 = [f for f in fastq_files if "_R1" in f or "_1.f" in f]
r2 = [f for f in fastq_files if "_R2" in f or "_2.f" in f]

total_size_gb = sum(os.path.getsize(f) for f in fastq_files) / (1024**3)

print(f"\u2705 FASTQ dir:  {fastq_dir}")
print(f"   Files:     {len(fastq_files)} ({total_size_gb:.1f} GB)")
print(f"   R1/R2:     {len(r1)} / {len(r2)} reads")
if len(r1) != len(r2):
    print(f"   \u26a0\ufe0f  R1/R2 count mismatch \u2014 pipeline may still work if naming is non-standard")

print(f"\u2705 Output dir: {output_dir}")

In [0]:
sys.path.insert(0, "/Workspace/Users/andrew_forman@eisai.com/.bundle/genesis_workbench/core/prod/files/app")
from utils.nextflow_pipeline import build_samplesheet, ALIGNERS

samplesheet_path = os.path.join(output_dir, "samplesheet.csv")
sheet_result = build_samplesheet(fastq_dir, samplesheet_path, sample_name)

if "error" in sheet_result:
    raise RuntimeError(f"\u274c Samplesheet error: {sheet_result['error']}")

print(f"\u2705 Samplesheet: {sheet_result['samplesheet_path']}")
print(f"   Samples:     {sheet_result['n_samples']}")
print(f"   FASTQ pairs: {sheet_result['n_fastq_pairs']}")
print(f"   Aligner:     {ALIGNERS[aligner]['name']} \u2014 {ALIGNERS[aligner]['description']}")
print(f"\nSamplesheet contents:")
with open(samplesheet_path) as f:
    print(f.read())

In [0]:
def generate_eisai_config(output_dir: str, aligner: str, genome: str) -> str:
    """
    Generate eisai.config — the institutional Nextflow config overlay.

    This is the KEY to zero-fork customization. Process selectors let us
    override resources, containers, publishDir, and behavior per-process
    WITHOUT modifying any nf-core pipeline code.

    When nf-core/scrnaseq is upgraded:
    - If a process name changes → update the withName selector here
    - If a new process is added → optionally add a selector
    - If a process is removed → the selector is harmlessly ignored

    This config is passed via: nextflow run ... -c eisai.config
    """
    config = f"""\
// =================================================================
// eisai.config — Institutional overlay for nf-core/scrnaseq
// Generated: {datetime.now(timezone.utc).isoformat()}
// Pattern:   Sandwich Wrapper (zero-fork)
//
// This file is GENERATED per-run. Edit the generate_eisai_config()
// function in the sandwich wrapper notebook, NOT this file.
// =================================================================

// ---- Global resource defaults ----
// Override per-process below. These are safety rails so no single
// process consumes the entire single-node cluster.
process {{
    // Default: generous but bounded
    cpus   = {{ Math.min(params.max_cpus ?: 8, Runtime.runtime.availableProcessors()) }}
    memory = {{ Math.min((params.max_memory ?: '48.GB') as nextflow.util.MemoryUnit, 
                        new nextflow.util.MemoryUnit(Runtime.runtime.maxMemory())) }}
    time   = '4.h'

    // Retry strategy: bump resources on transient failures
    errorStrategy = {{ task.exitStatus in [104, 134, 137, 139, 143, 247] ? 'retry' : 'finish' }}
    maxRetries    = 2

    // ---- Process-specific overrides ----
    // These use nf-core process names. If a name changes across versions,
    // update the selector — the rest of the pipeline is untouched.

    // STARsolo is the most resource-hungry process
    withName: 'STARSOLO' {{
        cpus   = {{ Math.min(12, Runtime.runtime.availableProcessors()) }}
        memory = {{ task.attempt == 1 ? '48.GB' : '64.GB' }}
        time   = '8.h'
    }}

    // Salmon alevin / alevin-fry
    withName: 'SALMON_ALEVIN|ALEVIN_FRY' {{
        cpus   = {{ Math.min(8, Runtime.runtime.availableProcessors()) }}
        memory = '32.GB'
    }}

    // kallisto bustools
    withName: 'KALLISTO_BUSTOOLS' {{
        cpus   = {{ Math.min(8, Runtime.runtime.availableProcessors()) }}
        memory = '24.GB'
    }}

    // MultiQC — lightweight, just needs a bit of memory for report gen
    withName: 'MULTIQC' {{
        cpus   = 2
        memory = '4.GB'
    }}

    // Genome indexing — only runs on first invocation (cached after)
    withName: 'STAR_GENOMEGENERATE|SALMON_INDEX|KALLISTO_INDEX' {{
        cpus   = {{ Math.min(12, Runtime.runtime.availableProcessors()) }}
        memory = '48.GB'
        time   = '6.h'
    }}
}}

// ---- Execution trace for post-flight ingestion ----
trace {{
    enabled   = true
    file      = '{output_dir}/trace/execution_trace.txt'
    sep       = '\\t'
    fields    = 'task_id,hash,native_id,name,status,exit,submit,start,complete,duration,realtime,%cpu,%mem,rss,vmem,peak_rss,peak_vmem,rchar,wchar,read_bytes,write_bytes'
    overwrite = true
}}

timeline {{
    enabled   = true
    file      = '{output_dir}/trace/timeline.html'
    overwrite = true
}}

report {{
    enabled   = true
    file      = '{output_dir}/trace/report.html'
    overwrite = true
}}

dag {{
    enabled   = true
    file      = '{output_dir}/trace/dag.html'
    overwrite = true
}}

// ---- Singularity/Conda config ----
// nf-core pipelines ship containers for every process. On Databricks
// single-node clusters, conda is simplest (no Docker-in-Docker).
conda {{
    enabled    = true
    cacheDir   = '/tmp/nf-conda-cache'
    createTimeout = '1.h'
}}

// ---- Cleanup ----
// Remove work directory after successful completion to save Volume space
cleanup = true
"""
    config_path = os.path.join(output_dir, "eisai.config")
    os.makedirs(os.path.dirname(config_path), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "trace"), exist_ok=True)

    with open(config_path, "w") as f:
        f.write(config)

    return config_path


eisei_config_path = generate_eisai_config(output_dir, aligner, genome)
print(f"\u2705 Institutional config: {eisai_config_path}")
print(f"   Trace output:        {output_dir}/trace/")
print(f"\nConfig preview (first 30 lines):")
with open(eisai_config_path) as f:
    for i, line in enumerate(f):
        if i >= 30:
            print("   ...")
            break
        print(f"   {line}", end="")

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    TimestampType, DoubleType, BooleanType,
)

AUDIT_TABLE = "dhbl_discovery_us_dev.genesis_schema.nextflow_run_audit"

run_started_at = datetime.now(timezone.utc)

# Schema for the audit table (created on first run)
audit_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("project_name", StringType(), True),
    StructField("pipeline", StringType(), True),
    StructField("pipeline_version", StringType(), True),
    StructField("aligner", StringType(), True),
    StructField("genome", StringType(), True),
    StructField("chemistry", StringType(), True),
    StructField("fastq_dir", StringType(), True),
    StructField("output_dir", StringType(), True),
    StructField("n_fastq_files", IntegerType(), True),
    StructField("fastq_size_gb", DoubleType(), True),
    StructField("n_samples", IntegerType(), True),
    StructField("status", StringType(), True),  # running / succeeded / failed / qc_failed
    StructField("started_at", TimestampType(), True),
    StructField("completed_at", TimestampType(), True),
    StructField("elapsed_minutes", DoubleType(), True),
    StructField("n_cells", IntegerType(), True),
    StructField("median_genes_per_cell", IntegerType(), True),
    StructField("qc_gate_passed", BooleanType(), True),
    StructField("scanpy_triggered", BooleanType(), True),
    StructField("error_message", StringType(), True),
    StructField("wrapper_version", StringType(), True),
])

# Insert initial "running" record
initial_row = Row(
    run_id=run_id,
    project_name=project_name,
    pipeline="nf-core/scrnaseq",
    pipeline_version=pipeline_version,
    aligner=aligner,
    genome=genome,
    chemistry=chemistry,
    fastq_dir=fastq_dir,
    output_dir=output_dir,
    n_fastq_files=len(fastq_files),
    fastq_size_gb=round(total_size_gb, 2),
    n_samples=sheet_result["n_samples"],
    status="running",
    started_at=run_started_at,
    completed_at=None,
    elapsed_minutes=None,
    n_cells=None,
    median_genes_per_cell=None,
    qc_gate_passed=None,
    scanpy_triggered=None,
    error_message=None,
    wrapper_version="1.0.0",
)

try:
    df = spark.createDataFrame([initial_row], schema=audit_schema)
    df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(AUDIT_TABLE)
    print(f"\u2705 Run registered: {AUDIT_TABLE}")
    print(f"   run_id: {run_id}")
    print(f"   status: running")
except Exception as e:
    # Non-fatal: audit failure shouldn't block the pipeline
    print(f"\u26a0\ufe0f  Audit table write failed (non-fatal): {e}")
    print(f"   Pipeline will continue without audit trail.")

print(f"\n{'='*60}")
print(f"  Pre-flight complete. Handing off to nf-core/scrnaseq.")
print(f"{'='*60}")

---
## 🟢 Layer 2 — nf-core/scrnaseq (Black Box)

The pipeline runs as an **opaque subprocess**. The only interface points are:
- `-c eisai.config` (institutional overlay from Layer 1)
- `--input`, `--outdir`, `--aligner`, `--genome` (standard nf-core params)

**ZERO modifications** to the pipeline itself. If this cell needs changing, it should only be to update the version pin.

In [0]:
from utils.nextflow_pipeline import build_nextflow_command
import time as _time

# Build the standard nf-core command
cmd = build_nextflow_command(
    samplesheet_path=samplesheet_path,
    output_dir=output_dir,
    aligner=aligner,
    genome=genome,
    chemistry=chemistry,
    pipeline_version=pipeline_version,
    extra_args=extra_args,
)

# Inject the institutional config overlay — the ONLY non-standard addition
# This goes AFTER the pipeline name so it overrides nf-core defaults
cmd_with_config = []
for i, arg in enumerate(cmd):
    cmd_with_config.append(arg)
    # Insert -c eisai.config right after the pipeline specifier
    if arg.startswith("nf-core/"):
        cmd_with_config.extend(["-c", eisai_config_path])

print(f"\ud83d\ude80 Running nf-core/scrnaseq v{pipeline_version}")
print(f"   Aligner: {aligner} | Genome: {genome}")
print(f"   Command:")
print(f"   {' '.join(cmd_with_config)}")
print(f"   Started: {_time.strftime('%Y-%m-%d %H:%M:%S UTC', _time.gmtime())}")
print(f"\n{'='*60}\n")

# Run with live output streaming
log_path = os.path.join(output_dir, "nextflow_run.log")
start_time = _time.time()

process = subprocess.Popen(
    cmd_with_config,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd=output_dir,
)

with open(log_path, "w") as log_file:
    for line in process.stdout:
        print(line, end="")
        log_file.write(line)

process.wait()
nf_elapsed = _time.time() - start_time
nf_exit_code = process.returncode

print(f"\n{'='*60}")
if nf_exit_code == 0:
    print(f"\u2705 Pipeline finished in {nf_elapsed/60:.1f} minutes")
else:
    print(f"\u274c Pipeline FAILED in {nf_elapsed/60:.1f} minutes (exit code: {nf_exit_code})")
    print(f"   Log: {log_path}")
    # Don't raise yet — let post-flight capture the failure in audit table

---
## 🟡 Layer 3 — Post-Flight (Databricks-Native)

Everything after `nextflow run` completes. Parses execution traces into Delta, runs QC gates on alignment quality, converts outputs to h5ad, and optionally triggers the downstream scanpy pipeline. **Zero nf-core modifications.**

In [0]:
import pandas as pd

TRACE_TABLE = "dhbl_discovery_us_dev.genesis_schema.nextflow_execution_traces"
trace_path = os.path.join(output_dir, "trace", "execution_trace.txt")

trace_df = None
if os.path.exists(trace_path):
    try:
        trace_pdf = pd.read_csv(trace_path, sep="\t")
        trace_pdf["run_id"] = run_id
        trace_pdf["project_name"] = project_name
        trace_pdf["pipeline_version"] = pipeline_version
        trace_pdf["aligner"] = aligner
        trace_pdf["ingested_at"] = datetime.now(timezone.utc)

        trace_df = spark.createDataFrame(trace_pdf)
        trace_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(TRACE_TABLE)

        print(f"\u2705 Execution trace: {len(trace_pdf)} tasks \u2192 {TRACE_TABLE}")
        print(f"\nProcess summary:")
        summary = trace_pdf.groupby("name").agg(
            count=("task_id", "count"),
            avg_duration=("realtime", lambda x: x.mean() if x.dtype != object else "N/A"),
            status_counts=("status", lambda x: dict(x.value_counts())),
        )
        for _, row in summary.iterrows():
            print(f"   {row.name}: {row['count']} tasks, status={row['status_counts']}")
    except Exception as e:
        print(f"\u26a0\ufe0f  Trace ingestion failed (non-fatal): {e}")
else:
    print(f"\u26a0\ufe0f  No execution trace found at {trace_path}")
    if nf_exit_code != 0:
        print(f"   (Expected \u2014 pipeline failed before producing trace)")

In [0]:
from utils.nextflow_pipeline import find_pipeline_outputs, convert_mtx_to_h5ad

# Initialize output tracking
pipeline_outputs = {"h5ad_files": [], "count_matrices": [], "multiqc_report": None, "raw_matrices": []}
n_cells_total = 0
median_genes = 0

if nf_exit_code == 0:
    pipeline_outputs = find_pipeline_outputs(output_dir, aligner)

    print(f"\ud83d\udce6 Pipeline Outputs:")
    print(f"   Count matrices (filtered): {len(pipeline_outputs['count_matrices'])}")
    print(f"   h5ad files:                {len(pipeline_outputs['h5ad_files'])}")
    print(f"   MultiQC report:            {pipeline_outputs['multiqc_report'] or 'not found'}")

    # Convert MTX to h5ad if needed
    if not pipeline_outputs["h5ad_files"]:
        mtx_dirs = set()
        for f in pipeline_outputs["count_matrices"]:
            if "filtered" in f:
                mtx_dirs.add(os.path.dirname(f))

        if mtx_dirs:
            print(f"\n\ud83d\udd04 Converting {len(mtx_dirs)} count matrices to h5ad...")
            for mtx_dir in mtx_dirs:
                sample = os.path.basename(os.path.dirname(mtx_dir)) or "sample"
                h5ad_path = os.path.join(output_dir, f"{sample}_filtered.h5ad")
                result = convert_mtx_to_h5ad(mtx_dir, h5ad_path)
                if result["status"] == "success":
                    print(f"   \u2705 {h5ad_path} ({result['n_cells']:,} cells \u00d7 {result['n_genes']:,} genes)")
                    pipeline_outputs["h5ad_files"].append(h5ad_path)
                    n_cells_total += result["n_cells"]
                else:
                    print(f"   \u274c {sample}: {result['error']}")
    else:
        # Read cell counts from existing h5ad files
        try:
            import scanpy as sc
            for h5ad_path in pipeline_outputs["h5ad_files"]:
                adata = sc.read_h5ad(h5ad_path, backed="r")
                n_cells_total += adata.n_obs
                print(f"   \ud83d\udcc4 {os.path.basename(h5ad_path)}: {adata.n_obs:,} cells \u00d7 {adata.n_vars:,} genes")
                adata.file.close()
        except Exception as e:
            print(f"   \u26a0\ufe0f  Could not read h5ad for cell counts: {e}")

    # Set task value for downstream jobs
    if pipeline_outputs["h5ad_files"]:
        h5ad_path_out = pipeline_outputs["h5ad_files"][0]
        dbutils.jobs.taskValues.set(key="h5ad_path", value=h5ad_path_out)
        print(f"\n\ud83d\udccb Task value: h5ad_path = {h5ad_path_out}")
else:
    print(f"\u26a0\ufe0f  Skipping output collection \u2014 pipeline failed (exit code {nf_exit_code})")

In [0]:
qc_passed = True
qc_reasons = []

if nf_exit_code != 0:
    qc_passed = False
    qc_reasons.append(f"Pipeline failed with exit code {nf_exit_code}")

elif qc_gate_enabled and pipeline_outputs["h5ad_files"]:
    print(f"\ud83d\udd0d Running alignment QC gates...")
    print(f"   Thresholds: min_cells={min_cells}, min_median_genes={min_median_genes}")

    try:
        import scanpy as sc
        import numpy as np

        for h5ad_path in pipeline_outputs["h5ad_files"]:
            adata = sc.read_h5ad(h5ad_path)
            sample_label = os.path.basename(h5ad_path)

            # Gate 1: minimum cell count
            if adata.n_obs < min_cells:
                qc_passed = False
                qc_reasons.append(
                    f"{sample_label}: {adata.n_obs} cells < {min_cells} minimum"
                )
                print(f"   \u274c {sample_label}: {adata.n_obs} cells (min: {min_cells})")
            else:
                print(f"   \u2705 {sample_label}: {adata.n_obs:,} cells (min: {min_cells})")

            # Gate 2: median genes per cell
            if hasattr(adata.X, "toarray"):
                genes_per_cell = (adata.X.toarray() > 0).sum(axis=1)
            else:
                genes_per_cell = (adata.X > 0).sum(axis=1)

            median_genes = int(np.median(genes_per_cell))
            if median_genes < min_median_genes:
                qc_passed = False
                qc_reasons.append(
                    f"{sample_label}: median {median_genes} genes/cell < {min_median_genes} minimum"
                )
                print(f"   \u274c {sample_label}: median {median_genes} genes/cell (min: {min_median_genes})")
            else:
                print(f"   \u2705 {sample_label}: median {median_genes} genes/cell (min: {min_median_genes})")

            # Gate 3: check for extremely high mitochondrial content (common failure mode)
            mito_genes = adata.var_names.str.startswith(("MT-", "mt-"))
            if mito_genes.any():
                if hasattr(adata.X, "toarray"):
                    mito_pct = adata[:, mito_genes].X.toarray().sum() / adata.X.toarray().sum() * 100
                else:
                    mito_pct = adata[:, mito_genes].X.sum() / adata.X.sum() * 100

                if mito_pct > 50:
                    qc_passed = False
                    qc_reasons.append(f"{sample_label}: {mito_pct:.1f}% mitochondrial (>50% indicates dead cells)")
                    print(f"   \u274c {sample_label}: {mito_pct:.1f}% mitochondrial (>50%)")
                else:
                    print(f"   \u2705 {sample_label}: {mito_pct:.1f}% mitochondrial")

    except ImportError:
        print("   \u26a0\ufe0f  scanpy not available \u2014 skipping QC gates")
        qc_reasons.append("scanpy not available for QC")
    except Exception as e:
        print(f"   \u26a0\ufe0f  QC gate error (non-fatal): {e}")
        qc_reasons.append(f"QC error: {str(e)[:200]}")

elif not qc_gate_enabled:
    print("\u23ed\ufe0f  QC gates disabled")

print(f"\n{'='*60}")
if qc_passed:
    print(f"\u2705 QC PASSED \u2014 alignment quality meets thresholds")
else:
    print(f"\u274c QC FAILED")
    for reason in qc_reasons:
        print(f"   \u2022 {reason}")

In [0]:
run_completed_at = datetime.now(timezone.utc)
run_elapsed = (run_completed_at - run_started_at).total_seconds() / 60

final_status = (
    "succeeded" if nf_exit_code == 0 and qc_passed
    else "qc_failed" if nf_exit_code == 0 and not qc_passed
    else "failed"
)

# Update audit table with final status
try:
    spark.sql(f"""
        MERGE INTO {AUDIT_TABLE} AS t
        USING (SELECT '{run_id}' AS run_id) AS s
        ON t.run_id = s.run_id
        WHEN MATCHED THEN UPDATE SET
            t.status = '{final_status}',
            t.completed_at = timestamp '{run_completed_at.isoformat()}',
            t.elapsed_minutes = {run_elapsed:.1f},
            t.n_cells = {n_cells_total or 'NULL'},
            t.median_genes_per_cell = {median_genes or 'NULL'},
            t.qc_gate_passed = {str(qc_passed).lower()},
            t.scanpy_triggered = false,
            t.error_message = {f"'{'; '.join(qc_reasons)[:500]}'" if qc_reasons else 'NULL'}
    """)
    print(f"\u2705 Audit table updated: status={final_status}")
except Exception as e:
    print(f"\u26a0\ufe0f  Audit update failed (non-fatal): {e}")

# Trigger downstream scanpy if enabled and QC passed
scanpy_job_triggered = False
if trigger_scanpy and qc_passed and pipeline_outputs["h5ad_files"]:
    try:
        from databricks.sdk import WorkspaceClient

        w = WorkspaceClient()
        scanpy_notebook = (
            "/Users/andrew_forman@eisai.com/genesis-workbench"
            "/modules/single_cell/scanpy/scanpy_v0.0.1/notebooks/analyze_single_h5ad"
        )

        scanpy_run = w.jobs.submit(
            run_name=f"Scanpy auto-trigger: {run_id}",
            tasks=[{
                "task_key": "scanpy_analysis",
                "notebook_task": {
                    "notebook_path": scanpy_notebook,
                    "base_parameters": {
                        "h5ad_path": pipeline_outputs["h5ad_files"][0],
                        "project_name": project_name,
                    },
                    "source": "WORKSPACE",
                },
                "new_cluster": {
                    "spark_version": "16.4.x-scala2.12",
                    "node_type_id": "r5.2xlarge",
                    "num_workers": 0,
                    "spark_conf": {
                        "spark.master": "local[*]",
                        "spark.databricks.cluster.profile": "singleNode",
                    },
                    "data_security_mode": "SINGLE_USER",
                },
            }],
        )

        scanpy_job_triggered = True
        print(f"\n\ud83d\ude80 Scanpy auto-triggered!")
        print(f"   Run ID: {scanpy_run.run_id}")
        print(f"   h5ad:   {pipeline_outputs['h5ad_files'][0]}")

        # Update audit table
        try:
            spark.sql(f"""
                MERGE INTO {AUDIT_TABLE} AS t
                USING (SELECT '{run_id}' AS run_id) AS s
                ON t.run_id = s.run_id
                WHEN MATCHED THEN UPDATE SET t.scanpy_triggered = true
            """)
        except Exception:
            pass

    except Exception as e:
        print(f"\u26a0\ufe0f  Scanpy trigger failed (non-fatal): {e}")

elif trigger_scanpy and not qc_passed:
    print(f"\n\u23ed\ufe0f  Scanpy NOT triggered \u2014 QC gates failed")
elif not trigger_scanpy:
    print(f"\n\u23ed\ufe0f  Scanpy auto-trigger disabled")

In [0]:
# =============================================================================
# Lineage Logging — Metadata Tier Integration
# Records assets + edges in the genesis_schema metadata tier tables so that
# downstream consumers (scanpy analysis, cell type annotation) can trace
# provenance back to the original FASTQs and alignment step.
# =============================================================================

try:
    from genesis_workbench.lineage import LineageLogger

    user_email = (
        dbutils.notebook.entry_point.getDbutils()
        .notebook().getContext().userName().get()
    )

    lineage = LineageLogger(
        module="single_cell",
        run_id=run_id,
        user_email=user_email,
        run_source="nextflow",
        catalog="dhbl_discovery_us_dev",
        schema="genesis_schema",
    )

    # ── Register input asset: FASTQs (bronze) ──
    fastq_asset = lineage.register_asset(
        path=fastq_dir,
        asset_type="volume_file",
        tier="bronze",
        format="fastq.gz",
        display_name=f"scrnaseq Input FASTQs ({project_name})",
        description=f"10X Genomics {chemistry} FASTQs for nf-core/scrnaseq alignment",
        tags={
            "genome": genome,
            "chemistry": chemistry,
            "aligner": aligner,
            "project_name": project_name,
        },
    )

    # ── Register output assets: h5ad files (gold — analysis-ready) ──
    h5ad_assets = []
    if nf_exit_code == 0 and pipeline_outputs.get("h5ad_files"):
        for h5ad_path in pipeline_outputs["h5ad_files"][:5]:  # Cap at 5
            h5ad_asset = lineage.register_asset(
                path=h5ad_path,
                asset_type="nextflow_output",
                tier="gold",
                format="h5ad",
                display_name=f"Gene Count h5ad ({os.path.basename(h5ad_path)})",
                description=f"Aligned count matrix via {aligner}, ready for scanpy",
                tags={
                    "aligner": aligner,
                    "genome": genome,
                    "chemistry": chemistry,
                    "pipeline": "nf-core/scrnaseq",
                    "pipeline_version": pipeline_version,
                    "n_cells": str(n_cells_total or 0),
                    "median_genes_per_cell": str(median_genes or 0),
                },
                row_count=n_cells_total,
            )
            h5ad_assets.append(h5ad_asset)

    # ── Register output assets: raw count matrices (gold) ──
    mtx_assets = []
    if nf_exit_code == 0 and pipeline_outputs.get("count_matrices"):
        for mtx_path in pipeline_outputs["count_matrices"][:3]:
            mtx_asset = lineage.register_asset(
                path=mtx_path,
                asset_type="nextflow_output",
                tier="gold",
                format="tsv",
                display_name=f"Count Matrix ({os.path.basename(mtx_path)})",
                tags={"aligner": aligner, "genome": genome},
            )
            mtx_assets.append(mtx_asset)

    # ── Record lineage edges ──
    # FASTQs → h5ad (consumed_by)
    for h5ad_asset in h5ad_assets:
        lineage.link(
            source=fastq_asset,
            target=h5ad_asset,
            relationship="consumed_by",
            step=f"{aligner}_alignment",
        )

    # FASTQs → count matrices (consumed_by)
    for mtx_asset in mtx_assets:
        lineage.link(
            source=fastq_asset,
            target=mtx_asset,
            relationship="consumed_by",
            step=f"{aligner}_quantification",
        )

    # If scanpy was triggered, record the trigger edge from h5ad → scanpy
    if scanpy_job_triggered and h5ad_assets:
        # Register a placeholder for the scanpy output (actual output logged by scanpy notebook)
        scanpy_placeholder = lineage.register_asset(
            path=f"mlflow-artifacts://scanpy_{run_id}",
            asset_type="mlflow_artifact",
            tier="platinum",
            display_name=f"Scanpy Analysis ({project_name})",
            description="Downstream scanpy pipeline (auto-triggered)",
            tags={"triggered_from": "scrnaseq_wrapper", "source_run": run_id},
        )
        lineage.link(
            source=h5ad_assets[0],
            target=scanpy_placeholder,
            relationship="triggered_by",
            step="auto_trigger_scanpy",
        )

    # ── Register the run ──
    lineage.register_run(
        run_name=f"scrnaseq_{project_name}",
        experiment_name="nf-core/scrnaseq",
        status=final_status,
        parameters={
            "fastq_dir": fastq_dir,
            "output_dir": output_dir,
            "aligner": aligner,
            "genome": genome,
            "chemistry": chemistry,
            "pipeline_version": pipeline_version,
        },
        metrics={
            "elapsed_minutes": run_elapsed,
            "n_cells": float(n_cells_total or 0),
            "median_genes_per_cell": float(median_genes or 0),
            "n_h5ad_files": float(len(pipeline_outputs.get("h5ad_files", []))),
        },
        start_time=run_started_at,
        end_time=run_completed_at,
        tags={
            "project_name": project_name,
            "qc_passed": str(qc_passed),
            "scanpy_triggered": str(scanpy_job_triggered),
            "aligner": aligner,
            "chemistry": chemistry,
        },
    )

    # ── Flush to Delta ──
    result = lineage.flush()
    print(f"\n\U0001f4cb Lineage logged: {result['assets_written']} assets, "
          f"{result['edges_written']} edges, {result['run_written']} run")

except ImportError:
    print("\u26a0\ufe0f  genesis_workbench.lineage not available — skipping lineage logging")
    print("   Install wheel v0.1.3+ to enable metadata tier integration")
except Exception as e:
    # Non-fatal: lineage logging should never break the alignment pipeline
    print(f"\u26a0\ufe0f  Lineage logging failed (non-fatal): {e}")

In [0]:
print("\u2554" + "\u2550"*58 + "\u2557")
print("\u2551  Sandwich Wrapper \u2014 Run Complete" + " "*24 + "\u2551")
print("\u2560" + "\u2550"*58 + "\u2563")
print(f"\u2551  Run ID:       {run_id:<42}\u2551")
print(f"\u2551  Pipeline:     nf-core/scrnaseq v{pipeline_version:<26}\u2551")
print(f"\u2551  Aligner:      {aligner:<42}\u2551")
print(f"\u2551  Genome:       {genome:<42}\u2551")
print(f"\u2551  Duration:     {run_elapsed:.1f} min (NF: {nf_elapsed/60:.1f} min){' '*(22-len(f'{run_elapsed:.1f} min (NF: {nf_elapsed/60:.1f} min)'))}\u2551")
print("\u2560" + "\u2550"*58 + "\u2563")
print(f"\u2551  Status:       {final_status.upper():<42}\u2551")
print(f"\u2551  Cells:        {n_cells_total or 'N/A'!s:<42}\u2551")
print(f"\u2551  QC gates:     {'PASSED' if qc_passed else 'FAILED':<42}\u2551")
print(f"\u2551  Scanpy:       {'Triggered' if scanpy_job_triggered else 'Not triggered':<42}\u2551")
print("\u255a" + "\u2550"*58 + "\u255d")

if qc_reasons:
    print(f"\nQC issues:")
    for r in qc_reasons:
        print(f"  \u2022 {r}")

if pipeline_outputs["h5ad_files"]:
    print(f"\nh5ad output(s):")
    for f in pipeline_outputs["h5ad_files"]:
        print(f"  \ud83d\udcc4 {f}")

print(f"\nArtifacts:")
print(f"  Log:       {log_path}")
print(f"  Config:    {eisai_config_path}")
print(f"  Trace:     {output_dir}/trace/")
if pipeline_outputs.get('multiqc_report'):
    print(f"  MultiQC:   {pipeline_outputs['multiqc_report']}")

# Fail the notebook cell if pipeline failed (so Jobs API sees it)
if nf_exit_code != 0:
    raise RuntimeError(f"nf-core/scrnaseq failed with exit code {nf_exit_code}. See log: {log_path}")

---
## Customization Guide

### Adding Custom Behavior (Zero nf-core Modifications)

All customization flows through `generate_eisai_config()` in the pre-flight layer:

| I want to... | How | Example |
|---|---|---|
| Tune resources for a process | Add `withName:` block in config | `withName: 'STARSOLO' { memory = '96.GB' }` |
| Use a custom container | Override `container` in selector | `withName: 'MULTIQC' { container = 'ecr.aws/eisai/multiqc:1.15' }` |
| Route outputs to a specific Volume | Add `publishDir` override | `withName: 'STARSOLO' { publishDir = [path: '/Volumes/.../star_out'] }` |
| Skip a step | Use nf-core `--skip_*` params | `extra_args = "--skip_multiqc"` (passed as widget) |
| Add a custom QC gate | Edit the post-flight QC cell | Add a new gate in the `qc_gate_enabled` block |
| Inject custom metadata | Edit the audit table schema | Add columns to `audit_schema` in pre-flight |
| Change the downstream trigger | Edit post-flight trigger cell | Point to a different notebook or add conditions |

### Version Upgrade Workflow

```
1. Check nf-core changelog:
   https://github.com/nf-core/scrnaseq/releases

2. Review for breaking changes:
   - Renamed processes? → Update withName selectors in generate_eisai_config()
   - Renamed params?    → Update build_nextflow_command() in nextflow_pipeline.py
   - New processes?     → Optionally add resource selectors
   - Removed processes? → Selectors are harmlessly ignored

3. Bump the version pin:
   - Default widget value in this notebook: pipeline_version = "X.Y.Z"
   - Default in run_nextflow_scrnaseq: same
   - Streamlit dropdown in single_cell.py: same

4. Test with a small sample:
   - Set compute_tier = "small"
   - Run a single-sample study
   - Verify QC gates pass
   - Check execution traces in Delta

5. Roll out:
   - Update bundle: databricks bundle deploy --target prod
   - Streamlit app picks up new defaults on next load
```

### Delta Tables Created

| Table | Purpose | Updated By |
|---|---|---|
| `genesis_schema.nextflow_run_audit` | Run metadata, status, QC results | Pre-flight (insert) + Post-flight (merge) |
| `genesis_schema.nextflow_execution_traces` | Per-process resource usage from NF trace | Post-flight (append) |

These tables enable operational dashboards (runs per week, failure rates, resource utilization trends) without touching nf-core.